# 03 · Qualitative rollouts (Figure 2, Active Matter appendix figure)

Set `well_dataset_name` in the first cell, run all. Needs `data/datasets/<pde>` (test split) and
`experiments/<pde>/results/<MODEL>_<pde>_traj.pkl` (one-sample rollouts) for every row in `models_to_plot_`.
Saves `artifacts/figures/fig_qualitative_<pde>.pdf`.


In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from einops import rearrange
import matplotlib.cm as cm
import numpy as np
import pickle
import torch
import math
import sys

from pathlib import Path as _Path
PROJECT_ROOT = str(next(p for p in [_Path.cwd(), *_Path.cwd().parents] if (p / "pyproject.toml").exists()))  # repo root, works on laptop and cluster
sys.path.insert(0, PROJECT_ROOT)

from wavelet_flow_matching.utils.fourier import isotropic_power_spectrum
from wavelet_flow_matching.utils.well_data import get_well_dataset
from wavelet_flow_matching.utils.plot import draw_movie, plot_psd, field2rgb
from wavelet_flow_matching.utils.rollout import get_gt_frame, get_pred_frame
from wavelet_flow_matching.utils.eval import build_stats_df

import matplotlib as mpl
mpl.rcParams.update({
    "text.usetex"         : True,
    "font.family"         : "serif",
    "font.serif"          : ["Computer Modern Roman"],
    "axes.labelsize"      : 15,
    "axes.titlesize"      : 15,
    "xtick.labelsize"     : 12,
    "ytick.labelsize"     : 12,
    "legend.fontsize"     : 12,
    "text.latex.preamble" : r"\usepackage{amsmath}",  # for \text, \mathbf, etc.
})

#well_dataset_name = 'turbulent_radiative_layer_2D'
well_dataset_name = 'rayleigh_benard'
#well_dataset_name = 'planetswe'
#well_dataset_name = 'active_matter'

RESULT_DIR = f"{PROJECT_ROOT}/experiments/{well_dataset_name}/results"

In [ ]:
well_base_path = f"{PROJECT_ROOT}/data/datasets"

dataset = get_well_dataset(
    path=well_base_path,
    physics=well_dataset_name,
    split='test',
    steps=-1,
    transform = ['log_scalars'],   
    augment = [],
)

meta_fields = dataset.metadata.field_names
field_names = [item for sublist in meta_fields.values() for item in sublist]

In [ ]:
import os
from wavelet_flow_matching.analysis import catalog as C

# Rows of the qualitative figure (Figure 2 for RB and TRL, the AM figure in Appendix E).
# Keys are the paper model names used for experiments/<pde>/results/<MODEL>_<pde>_traj.pkl;
# WFM rows use the wavelet selected for this PDE (Table "optimal wavelets").
SEL = C.BEST_WFM_PER_PDE[well_dataset_name]
models_to_plot_ = {
    "FM_pixel"                         : r"FM$_{pixel}$",
    f"WFM_1scale_{SEL['1scale']}"      : r"WFM 1-scale",
    f"WFM_3scale_{SEL['3scale']}"      : r"WFM 3-scale",
    #"FM_fourier"                      : r"FM$_{fourier}$",
    #"LatentFM"                        : r"LatentFM$_{\div 64}$",   # RB only
    #"WDNO"                            : r"WDNO",
    "FNO"                              : r"FNO",
    #"TFNO"                            : r"TFNO",
    #"WNO"                             : r"WNO",
}

traj_path = lambda m: f"{RESULT_DIR}/{m}_{well_dataset_name}_traj.pkl"
missing = [m for m in models_to_plot_ if not os.path.exists(traj_path(m))]
if missing:
    print("missing rollouts (run inference with n_ensemble=1 and rename with "
          "scripts/rename_to_paper_names.py):", missing)

# model data
rollout_traj = {}
for model in models_to_plot_:
    if model in missing:
        continue
    with open(traj_path(model), 'rb') as f:
        data = pickle.load(f)
    rollout_traj[model] = data['preds'] if isinstance(data, dict) else data
    print(f"Loaded rollout trajectory for {model}")
models_to_plot_ = {m: l for m, l in models_to_plot_.items() if m not in missing}

# ground truth data
with open(f'{RESULT_DIR}/ground_truth_{well_dataset_name}_traj.pkl', 'rb') as f:
    truth_traj = pickle.load(f)
    print("Loaded ground truth trajectory")


In [ ]:
# Quick preview (rows = models). The paper figure is the next cell.
TRAJ_IDX        = 0
ENS_IDX         = 0
N_STEPS_INPUT   = 4
field_idx       = 0
field_name      = field_names[field_idx]
T_TOTAL         = dataset.metadata.n_steps_per_trajectory[0]
N_ROLLOUT_STEPS = T_TOTAL - N_STEPS_INPUT
N_TRAJ          = len(dataset.metadata.n_steps_per_trajectory)                  # number of trajectories in the test set for this PDE

#lead_times = [1, 16, 32, 48, 64, 80, 96]
#lead_times = [1, 16, 32, 48, 64, 80, 96]
#lead_times = [1, 32, 64, 96, 128, 160, 192]
#lead_times = [1, 10, 20, 30, 40, 45]
#lead_times = [1, 3, 6, 10, 16, 24]

lead_times = {
    "turbulent_radiative_layer_2D": [1, 32, 64, 96],
    "rayleigh_benard":              [1, 32, 96, 192],
    "active_matter":                [1, 8, 16, 32],
}[well_dataset_name]

gt_frames = [get_gt_frame(truth_traj, TRAJ_IDX, t, N_STEPS_INPUT, field_idx) for t in lead_times]
model_frames = {
    name: [get_pred_frame(rollout_traj[name], TRAJ_IDX, ENS_IDX, t, field_idx) for t in lead_times]
    for name in models_to_plot_.keys()
}

row_labels     = ["Ground-truth"] + list(models_to_plot_.values())
all_row_frames = [gt_frames] + [model_frames[name] for name in models_to_plot_.keys()]
n_rows, n_cols = len(row_labels), len(lead_times)

# ── Compute vmin/vmax from the GROUND TRUTH row only ─────────────────────────
gt_vals = np.concatenate([f.ravel() for f in gt_frames])

gt_vmin = float(np.percentile(gt_vals, 1))
gt_vmax = float(np.percentile(gt_vals, 99))
#gt_vmin = gt_vals.min()
#gt_vmax = gt_vals.max()


# ── Convert ALL frames to RGB using the ground truth limits ──────────────────
rgb_rows = [
    [field2rgb(frame, vmin=gt_vmin, vmax=gt_vmax, cmap="RdBu_r") for frame in row]
    for row in all_row_frames
]

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(0.7 * n_cols, 1.7 * n_rows),
    gridspec_kw={"hspace": 0.03, "wspace": 0.05},
)

if n_rows == 1:
    axes = axes[None, :]

plt.rcParams["axes.linewidth"] = 0.02

for ri, (rgb_frames, label) in enumerate(zip(rgb_rows, row_labels)):
    for ci, lead in enumerate(lead_times):
        ax = axes[ri, ci]
        ax.imshow(rgb_frames[ci], origin="lower", aspect="auto")
        ax.set_xticks([])
        ax.set_yticks([])
        if ci == 0:
            #ax.set_ylabel(rf'${label}$', fontsize=8) #, fontweight="bold")
            ax.set_ylabel(label, fontsize=8) #, fontweight="bold")
        if ri == 0:
            ax.set_title(rf"$t={lead}$", fontsize=8, fontweight="bold")

    # ── One colorbar per row, all using ground truth scale ───────────────────
    #sm = cm.ScalarMappable(norm=mcolors.Normalize(vmin=gt_vmin, vmax=gt_vmax), cmap="RdBu_r")
    #sm.set_array([])
    #cbar = fig.colorbar(sm, ax=axes[ri, -1], fraction=0.046, pad=0.04, label=field_name)
    #cbar.ax.tick_params(labelsize=7)

#plt.savefig(f"{PROJECT_ROOT}/artifacts/figures/fig_qualitative_{well_dataset_name}.pdf", dpi=300, bbox_inches='tight', transparent=False)
plt.show()

In [ ]:
# Paper figure: Figure 2 (RB, TRL) and the Active Matter figure in Appendix E.
TRAJ_IDX        = 0
ENS_IDX         = 0
N_STEPS_INPUT   = 4
field_idx       = 0
field_name      = field_names[field_idx]
T_TOTAL         = dataset.metadata.n_steps_per_trajectory[0]
N_ROLLOUT_STEPS = T_TOTAL - N_STEPS_INPUT
N_TRAJ          = len(dataset.metadata.n_steps_per_trajectory)
TEXT_WIDTH      = 20                           # inches — match your LaTeX \textwidth
CELL_ASPECT     = 0.5                          # height/width ratio per cell; tune to your field shape

lead_times = {
    "turbulent_radiative_layer_2D": [1, 32, 64, 96],
    "rayleigh_benard":              [1, 32, 96, 192],
    "active_matter":                [1, 8, 16, 32],
}[well_dataset_name]

REVERSE_LEAD_TIMES = True
if REVERSE_LEAD_TIMES:
    lead_times_display = lead_times[::-1]  # reversed for display
else:
    lead_times_display = lead_times        # keep original order for display

# ── Collect frames ────────────────────────────────────────────────────────────
gt_frames = [get_gt_frame(truth_traj, TRAJ_IDX, t, N_STEPS_INPUT, field_idx) for t in lead_times_display]
model_frames = {
    name: [get_pred_frame(rollout_traj[name], TRAJ_IDX, ENS_IDX, t, field_idx) for t in lead_times_display]
    for name in models_to_plot_.keys()
}
col_labels     = ["Ground-truth"] + list(models_to_plot_.values())
all_col_frames = [gt_frames] + [model_frames[name] for name in models_to_plot_.keys()]

# ── Compute vmin/vmax from ground truth only ──────────────────────────────────
gt_vals = np.concatenate([f.ravel() for f in gt_frames])
gt_vmin = float(np.percentile(gt_vals, 1))
gt_vmax = float(np.percentile(gt_vals, 99))

# ── Convert all frames to RGB ─────────────────────────────────────────────────
rgb_cols = [
    #[field2rgb(frame, vmin=gt_vmin, vmax=gt_vmax, cmap="RdBu_r") for frame in col]
    [field2rgb(frame, vmin=gt_vmin, vmax=gt_vmax, cmap="viridis") for frame in col]
    for col in all_col_frames
]

# ── Layout ────────────────────────────────────────────────────────────────────
n_rows = len(lead_times_display)
n_cols = len(col_labels)

fig_w = TEXT_WIDTH
fig_h = fig_w / n_cols * CELL_ASPECT * n_rows

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(fig_w, fig_h),
    gridspec_kw={"hspace": 0.1, "wspace": 0.08},
)

if n_rows == 1: axes = axes[None, :]
if n_cols == 1: axes = axes[:, None]

# ── Reserve margins for labels ────────────────────────────────────────────────
left_margin   = 0.08    # space for τ labels on the left
bottom_margin = 0.10    # space for model labels on the bottom
fig.subplots_adjust(left=left_margin, bottom=bottom_margin, right=0.99, top=0.99)

plt.rcParams["axes.linewidth"] = 0.02

# ── Draw images ───────────────────────────────────────────────────────────────
for ci, rgb_frames in enumerate(rgb_cols):
    for ri, lead in enumerate(lead_times_display):
        ax = axes[ri, ci]
        #ax.imshow(rgb_frames[ri].transpose(1, 0, 2), origin="lower", aspect="auto")
        ax.imshow(rgb_frames[ri], origin="lower", aspect="auto")
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values(): 
            spine.set_visible(False)

# ── Force layout so get_position() is accurate ───────────────────────────────
fig.canvas.draw()

# ── τ labels on the left ──────────────────────────────────────────────────────
for ri, lead in enumerate(lead_times_display):
    pos = axes[ri, 0].get_position()
    fig.text(
        x=pos.x0 - 0.01,
        y=(pos.y0 + pos.y1) / 2,
        s=rf"$t={lead}$",
        fontsize=24,             # ← scaled to physical figure size
        fontweight="bold",
        ha="right",
        va="center",
        rotation=90,
        transform=fig.transFigure,
    )

# ── Model labels on the bottom ────────────────────────────────────────────────
for ci, label in enumerate(col_labels):
    pos = axes[-1, ci].get_position()
    fig.text(
        x=(pos.x0 + pos.x1) / 2,
        y=pos.y0 - 0.03,
        #s=rf"${label}$",                # ← plain string; wrap in $...$ only if LaTeX needed
        s=label,
        fontsize=24,             # ← small enough to fit n_cols labels
        ha="center",
        va="top",
        rotation=0,            # ← angled so long model names don't overlap
        transform=fig.transFigure,
    )

plt.savefig(f"{PROJECT_ROOT}/artifacts/figures/fig_qualitative_{well_dataset_name}.pdf", dpi=400, bbox_inches="tight", transparent=False)
plt.show()